# Module 10: Explore Expanded Inventory Signals and Check Sensitivity

目标：用 Module 9 的扩大配对样本，观察库存增长差与下一年财务表现的关联，以及关联是否随公司、子行业或样本规则改变。
仍然是 8 个 Code Cells：中文教学说明，英文代码、注释、列名、图表与文件名。
使用 pandas、matplotlib、普通循环与条件判断；没有自定义分析函数，不下载数据，不引入回归模型。

接在现有 Notebook 的 Module 9 后面，按顺序运行下面 8 格；也可以单独运行 Guide，使用前面相同的 PROJECT_ROOT。
当前实际目录是 `[project root]`，不是 Downloads。
输入：expanded_gm_next_year_pairs.csv、expanded_ocf_next_year_pairs.csv、expanded_module9_metadata.json。
输出：样本覆盖、分组描述、子行业差异、公司内部比较、相关性敏感性表、两张报告用 PNG 图及处理说明。

研究预期：如果库存增速超过销售增速预示经营压力，则 Gap > 0 组下一年的比率变化应更低。
这里让数据检验这个方向，不根据结果修改阈值；相反或不稳定的结果也是报告发现。
毛利与现金流分别使用自己的合格样本，不把两个表按行号拼在一起，也不强行先缩成同样大小。

观察单位是公司年度配对，不是独立公司。较长历史的公司在全样本平均中占更多权重。
本模块补充公司内部比较和公司等权汇总，但这些仍是描述性检查，不处理全部共同年度冲击、时间相关性或混杂因素。
不计算 p 值、显著性或因果效应，不把历史下降比例当成未来预测概率。
沿用 Module 9 的 provisional 状态、历史会计审核限制和年报提交时间边界。


## Cell 1: Set Up the Analysis and Figure Directory

复用已存在的 PROJECT_ROOT；单独运行时使用当前工作目录。
如果打印路径不对，在导入语句后加 `PROJECT_ROOT = Path("[project root]")`，再重跑本格。
PROCESSED_DIR 读取已有 CSV，FIGURE_DIR 保存 PNG。mkdir 不会清空已有图表。
matplotlib 在 Module 6 已经用过，通常可以直接导入；只有提示未安装时，才另开一格执行 `%pip install matplotlib`。
plt.show() 会在 Notebook 展示图片，这里不需要额外的 Jupyter 指令。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["font.size"] = 10
print("Project directory:", PROJECT_ROOT)


## Cell 2: Read Separate Outcome Samples and Define Display Columns

settings 是字典：每个研究名称对应文件名、本期比率、下一年度变化、下降标签。
settings.items() 返回名称与对应的四项组合；for 后的括号把这四项分别赋给 filename / ratio / outcome / label。
循环每次读一个结果样本，再存进 analysis_tables。这样毛利与现金流复用同一套分析步骤，不混淆目标列。
你可以用 `analysis_tables["Gross margin"]` 单独查看毛利表。
assert 要求公司年度不重复、库存信号与本项结果不缺失；失败时返回 Module 9 查原因，不用填零绕过。

原数据用小数保存比率。新增展示列：
- inventory_gap_pp：库存增长差乘 100，单位为百分点。
- outcome_change_pp：下一年度比率变化乘 100，单位为百分点。
- current_ratio_pct / next_ratio_pct：本期和下一期比率乘 100，单位为百分比。
- declined：只复制本项结果的下降标签，不同时要求另一项结果存在。
- inventory_group：固定分为 Gap > 0 与 Gap <= 0，恰好等于零属于后一组。

common_keys 按 ticker + period_end 做 inner merge，得到两个结果都可用的同一批公司年度键。
共同样本只在敏感性检查使用；主分析仍用各自的全部合格样本。
当前应打印毛利 401 行 / 36 家，现金流 490 行 / 42 家，共同键 399 个。


In [ ]:
settings = {
    "Gross margin": ("expanded_gm_next_year_pairs.csv", "gross_margin", "next_gross_margin_change", "gross_margin_declined"),
    "OCF-to-revenue": ("expanded_ocf_next_year_pairs.csv", "ocf_margin", "next_ocf_margin_change", "ocf_margin_declined"),
}
analysis_tables = {}
for name, (filename, ratio, outcome, label) in settings.items():
    table = pd.read_csv(PROCESSED_DIR / filename, parse_dates=["period_end", "next_period_end"], dtype={label: "boolean"})
    assert not table.duplicated(["ticker", "period_end"]).any(), "Duplicate company-year pairs."
    assert table[["inventory_sales_gap", outcome, label]].notna().all().all(), "Review missing signals or outcomes."
    table["inventory_gap_pp"] = table["inventory_sales_gap"] * 100
    table["outcome_change_pp"] = table[outcome] * 100
    table["current_ratio_pct"] = table[ratio] * 100
    table["next_ratio_pct"] = table["next_" + ratio] * 100
    table["declined"] = table[label]
    table["inventory_group"] = "Gap <= 0"
    table.loc[table["inventory_sales_gap"].gt(0), "inventory_group"] = "Gap > 0"
    analysis_tables[name] = table
    print(name, "pairs:", len(table), "| companies:", table["ticker"].nunique())
common_keys = analysis_tables["Gross margin"][["ticker", "period_end"]].merge(
    analysis_tables["OCF-to-revenue"][["ticker", "period_end"]], on=["ticker", "period_end"], validate="one_to_one",
)
module9_metadata = json.loads((PROCESSED_DIR / "expanded_module9_metadata.json").read_text(encoding="utf-8"))
first_end = min(table["period_end"].min() for table in analysis_tables.values()).date()
last_end = max(table["period_end"].max() for table in analysis_tables.values()).date()
source_caption = f"Source: cached SEC Company Facts / Module 9 | Current ends: {first_end} to {last_end} | Provisional"
print("Common outcome pairs:", len(common_keys))


## Cell 3: Describe Coverage and Compare the Two Inventory Groups

coverage 统计每家公司贡献的年度数、首末期间及子行业，方便识别哪些公司历史较长。
pooled_summary 把本项结果的全部年度按库存组汇总。这里的 pooled 指合并所有公司年度；每条配对权重相同，不是每家公司权重相同。
.agg 的写法是 新列名=(输入列名, 汇总方法)，与前面的 Module 6 相同。
size 计算观察数，nunique 计算不同公司数量，mean 是平均值，median 是中位数。
declined 的均值是样本内下降比例：0.60 表示该组 60% 的配对出现下一年下降。

同时看 mean 和 median，避免仅靠容易被极端年度影响的均值。
预期方向是 Gap > 0 组变化更低、下降比例更高，但结果未必支持。
公司数有可能在两组重复：同一家公司不同年度可以出现在不同组，不能把两组公司数相加。
groupby 没出现的组不会被补为零；没有观察值意味着不能估计该组平均表现。

最后显示每个结果样本的信号和目标分布。describe 的 min/max 帮助定位极端值，但不能仅凭极端就认定数据错误。
后面的百分位敏感性检查只改变分析子样本，不删除原记录、不修改财务金额。


In [ ]:
coverage_tables = []
pooled_tables = []
for name, table in analysis_tables.items():
    coverage = table.groupby(["ticker", "subgroup"]).agg(
        pairs=("ticker", "size"), first_period_end=("period_end", "min"), last_period_end=("period_end", "max"),
    ).reset_index()
    coverage["outcome"] = name
    coverage_tables.append(coverage)
    summary = table.groupby("inventory_group").agg(
        pairs=("ticker", "size"), companies=("ticker", "nunique"),
        mean_change_pp=("outcome_change_pp", "mean"), median_change_pp=("outcome_change_pp", "median"),
        decline_rate=("declined", "mean"),
    ).reset_index()
    summary["outcome"] = name
    pooled_tables.append(summary)
    print(name, "signal and outcome distributions:")
    display(table[["inventory_gap_pp", "outcome_change_pp"]].describe().round(3))
sample_coverage = pd.concat(coverage_tables, ignore_index=True)
pooled_summary = pd.concat(pooled_tables, ignore_index=True)
display(pooled_summary.round(3))
display(sample_coverage.head(10))


## Cell 4: Plot Inventory Gaps against Next-Year Changes

每个点是一条公司年度配对，横轴是本期库存增长差，纵轴是下一年度比率变化，两个轴均为百分点。
不同子行业用不同颜色。使用短名称只为图例更易读，不修改数据里的原分组名称。
同一行业在两个面板使用同一颜色，便于比较；没有按结果挑颜色或删除极端点。
fig.legend 在绘图区外放置全图共用图例，避免图例盖住极端观察。
零线帮助判断方向。右下象限符合“库存增速超过销售、下一年比率下降”的组合，但散点出现在那里本身不是因果证据。

左图与右图来自不同结果样本，标题分别标注公司数和配对数；不能当作同一批点一一比较。
不添加拟合线或显著性标记，先观察点云、不同子行业和极端观察。
图片保留来源、当前年度终点范围及 provisional 提醒，保存为 expanded_inventory_gap_scatter.png。


In [ ]:
short_groups = {
    "Apparel, Footwear, and Off-Price Retail": "Apparel / footwear / off-price",
    "Department Stores": "Department stores",
    "Home Furnishings, Furniture, and Home Improvement": "Home / furniture / improvement",
    "Sporting Goods, Electronics, and Other Merchandise": "Sporting / electronics / other",
    "Beauty and Personal Care": "Beauty / personal care",
}
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5))
for ax, (name, table) in zip(axes, analysis_tables.items()):
    for group_index, (subgroup, short_name) in enumerate(short_groups.items()):
        group = table.loc[table["subgroup"].eq(subgroup)]
        ax.scatter(group["inventory_gap_pp"], group["outcome_change_pp"], color=f"C{group_index}", label=short_name, s=25, alpha=0.65)
    ax.axhline(0, color="gray", linewidth=0.8)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_xlabel("Inventory growth minus sales growth (pp)")
    ax.set_ylabel("Next-year ratio change (pp)")
    ax.set_title(f"{name}: {table['ticker'].nunique()} companies, {len(table)} pairs")
    ax.text(0.02, 0.98, f"Current ends: {table['period_end'].min().date()} to {table['period_end'].max().date()}", transform=ax.transAxes, va="top", fontsize=8)
fig.legend(*axes[0].get_legend_handles_labels(), loc="lower center", bbox_to_anchor=(0.5, 0.045), ncol=3, fontsize=8)
fig.text(0.5, 0.01, source_caption, ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.18, 1, 1])
fig.savefig(FIGURE_DIR / "expanded_inventory_gap_scatter.png", dpi=180, bbox_inches="tight")
plt.show()


## Cell 5: Compare Inventory Groups within Each Retail Subgroup

全样本均值可能混入子行业差异，因此改在每个 subgroup 内分别比较两组。
subgroup_summary 保留两组的配对数、公司数、均值、中位数和下降比例。
subgroup_differences 计算：mean_difference_pp = Gap > 0 组平均变化 - Gap <= 0 组平均变化。
例如 -0.80 表示正差值组下一年变化低 0.80 个百分点；+0.80 代表方向相反。
这是两组平均变化的差，不是正差值组本身的平均变化，也不是因果效应。

unstack("inventory_group") 把分组行转成列；reindex 固定两组列顺序。
某个子行业没有一组时，对应均值与差异留为 NaN，不填零制造对照组。
正、非正组的计数保留下来，便于识别很小或不平衡的组。
chart.itertuples() 把图表工作表逐行读出，可以用 row.subgroup、row.positive_gap_pairs 取得该行字段，组成英文标签。
dropna 只跳过无法计算差异的画图项目，完整汇总表仍保留 NaN；没有把无对照组当成零差异。
图上每根横柱是一个子行业的组间均值差；横轴左侧符合原假设方向，右侧相反。
每行标签的 + / - 分别表示 Gap > 0 / Gap <= 0 的配对数，不是正负结果数量。

这是行业内描述，不保证行业内公司可互换；成本范围、门店模式和商业组合仍可能不同。
横柱没有误差区间，不能从它是否跨零判断统计显著性。


In [ ]:
subgroup_tables = []
difference_tables = []
for name, table in analysis_tables.items():
    summary = table.groupby(["subgroup", "inventory_group"]).agg(
        pairs=("ticker", "size"), companies=("ticker", "nunique"),
        mean_change_pp=("outcome_change_pp", "mean"), median_change_pp=("outcome_change_pp", "median"),
        decline_rate=("declined", "mean"),
    )
    means = summary["mean_change_pp"].unstack("inventory_group").reindex(columns=["Gap <= 0", "Gap > 0"])
    counts = summary["pairs"].unstack("inventory_group").reindex(columns=["Gap <= 0", "Gap > 0"])
    differences = pd.DataFrame({
        "mean_difference_pp": means["Gap > 0"] - means["Gap <= 0"],
        "positive_gap_pairs": counts["Gap > 0"], "nonpositive_gap_pairs": counts["Gap <= 0"],
    }).reset_index()
    summary = summary.reset_index()
    summary["outcome"] = name
    differences["outcome"] = name
    subgroup_tables.append(summary)
    difference_tables.append(differences)
subgroup_summary = pd.concat(subgroup_tables, ignore_index=True)
subgroup_differences = pd.concat(difference_tables, ignore_index=True)
display(subgroup_summary.round(3))
display(subgroup_differences.round(3))
fig, axes = plt.subplots(2, 1, figsize=(11, 8))
for ax, name in zip(axes, analysis_tables):
    chart = subgroup_differences.loc[subgroup_differences["outcome"].eq(name)].dropna(subset=["mean_difference_pp"])
    labels = [f"{short_groups[row.subgroup]} (+:{int(row.positive_gap_pairs)}, -:{int(row.nonpositive_gap_pairs)})" for row in chart.itertuples()]
    bars = ax.barh(labels, chart["mean_difference_pp"], color="tab:blue", alpha=0.8)
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_title(name)
    ax.set_xlabel("Mean next-year change: Gap > 0 minus Gap <= 0 (pp)")
    ax.set_ylabel("Retail subgroup")
    ax.margins(x=0.2)
fig.text(0.5, 0.01, source_caption + "\n+ / -: positive / nonpositive gap pair counts", ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.07, 1, 1])
fig.savefig(FIGURE_DIR / "expanded_subgroup_mean_differences.png", dpi=180, bbox_inches="tight")
plt.show()


## Cell 6: Compare Groups within Companies and Give Companies Equal Weight

不同公司经营模式和费用分类不同，所以再在同一家公司内部比较两组。
mean_difference_pp 的方向与上一格一致，只是对照范围从子行业缩到公司。
公司内部比较可以减少混入公司固定差异的问题，但不同年份的宏观环境和公司经营状态仍可能不同。

MIN_GROUP_PAIRS=3 是这里预先固定的小样本门槛：同一家公司两组各至少 3 条，才进入公司等权概要。
它只用于这一项概要，不改变 Module 9 的主样本，也不表示 3 条观察已经足以可靠推断。
某组不存在、或只有 1–2 条时，公司比较仍保存在工作表，comparison_usable 为 False。

mean_company_difference_pp 是合格公司差异的平均值，每家公司权重相同，而不是每条年度权重相同。
median_company_difference_pp 展示典型公司方向；share_negative 表示合格公司里有多少比例的均值差为负。
share_negative 不等于配对的下降比例，也不是未来表现概率。

当前预计毛利 31 家、现金流 36 家公司进入这个等权概要；公司数比主样本小是规则结果，不是遗漏。
如果公司等权结果与全年度均值不同，先考虑样本构成和权重差异，不要只选择支持原假设的一种结果。
没有合格公司时，比例留 NaN，不以 0 表示不存在关联。


In [ ]:
MIN_GROUP_PAIRS = 3
company_tables = []
balanced_rows = []
for name, table in analysis_tables.items():
    grouped = table.groupby(["ticker", "subgroup", "inventory_group"])["outcome_change_pp"].agg(["size", "mean"])
    means = grouped["mean"].unstack("inventory_group").reindex(columns=["Gap <= 0", "Gap > 0"])
    counts = grouped["size"].unstack("inventory_group").reindex(columns=["Gap <= 0", "Gap > 0"])
    comparison = pd.DataFrame({
        "mean_difference_pp": means["Gap > 0"] - means["Gap <= 0"],
        "positive_gap_pairs": counts["Gap > 0"], "nonpositive_gap_pairs": counts["Gap <= 0"],
    }).reset_index()
    comparison["comparison_usable"] = comparison["positive_gap_pairs"].ge(MIN_GROUP_PAIRS) & comparison["nonpositive_gap_pairs"].ge(MIN_GROUP_PAIRS)
    comparison["outcome"] = name
    company_tables.append(comparison)
    valid = comparison.loc[comparison["comparison_usable"], "mean_difference_pp"]
    balanced_rows.append({
        "outcome": name, "companies": len(valid), "mean_company_difference_pp": valid.mean(),
        "median_company_difference_pp": valid.median(),
        "share_negative": valid.lt(0).mean() if len(valid) else float("nan"),
    })
company_comparisons = pd.concat(company_tables, ignore_index=True)
company_balanced_summary = pd.DataFrame(balanced_rows)
display(company_balanced_summary.round(3))
display(company_comparisons.head(10).round(3))


## Cell 7: Check Correlations across Four Predefined Samples

这格检查相关方向是否依赖观察选择，而不是检验统计显著性。固定比较四种样本：
1. All pairs：每项结果的完整主样本。
2. Unchanged period lengths：排除本期/目标年度长度变化标记，用于 52/53 周等长度差异敏感性检查。
3. Common outcome pairs：仅保留毛利与现金流都有合格结果的相同公司年度。
4. Central 98% of inventory gaps：保留本项信号第 1–99 百分位区间的记录，观察极端库存差是否主导关系。

第 4 项是截尾敏感性，不是认定极端值有错；保留完整主样本，不修改原信号、不按结果大小删记录。
quantile(0.01 / 0.99) 只使用库存特征，各结果样本分别计算自己的边界，trim_limits 保存实际阈值。
边界采用 between，包括端点，有限样本下保留数量不一定精确等于 98%。
当前预计四种样本行数分别为：毛利 401 / 201 / 399 / 393；现金流 490 / 235 / 399 / 480。

pearson_change_r 衡量库存差与下一年变化的线性关联。
spearman_change_r 先用 rank() 把两列转换成排序名次，再计算名次间 Pearson 相关，观察单调关系；默认相同值采用平均名次。
任一变量没有变化时，相关系数留 NaN，不视为零关联。
company_demeaned_r 先从每个年度值减去本公司的样本均值，再计算相关，让我们观察去掉公司均值差异后的方向。
transform("mean") 会把公司的均值返回到每条原记录上，长度与原表一致；普通 groupby.mean() 则只返回每家公司一行。
这个操作不是已完成固定效应回归，也没有控制共同年度冲击或提供标准误。

**特别注意“变化”与“水平”：**
current_level_r 是信号与本期比率的相关，next_level_r 是信号与下一期比率的相关。
目标变化=下一期比率-本期比率，因此信号与较低本期现金流相关时，后续恢复也可能带来正的变化相关。
比较这三项能检查这种可能性；仅凭当前表不能确定均值回归或具体经营机制。
较高库存差对应正的现金流变化，不等于对应较高的下一年现金流水平，也不证明库存积累改善经营。
不要把同一公式构造的变化关联直接解释为独立的预测能力；后续应进一步控制基期比率并检查目标水平。

参考：[pandas corr](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html)、[pandas rank](https://pandas.pydata.org/docs/reference/api/pandas.Series.rank.html)。


In [ ]:
sensitivity_rows = []
trim_rows = []
for name, table in analysis_tables.items():
    lower = table["inventory_gap_pp"].quantile(0.01)
    upper = table["inventory_gap_pp"].quantile(0.99)
    trim_rows.append({"outcome": name, "lower_gap_pp": lower, "upper_gap_pp": upper})
    common = table.merge(common_keys, on=["ticker", "period_end"], validate="one_to_one")
    for sample_name, sample in [
        ("All pairs", table), ("Unchanged period lengths", table.loc[~table["pair_period_length_changed"]]),
        ("Common outcome pairs", common), ("Central 98% of inventory gaps", table.loc[table["inventory_gap_pp"].between(lower, upper)]),
    ]:
        x = sample["inventory_gap_pp"]
        y = sample["outcome_change_pp"]
        row = {"outcome": name, "sample": sample_name, "pairs": len(sample), "companies": sample["ticker"].nunique()}
        for column, result in [("outcome_change_pp", "pearson_change_r"), ("current_ratio_pct", "current_level_r"), ("next_ratio_pct", "next_level_r")]:
            row[result] = x.corr(sample[column]) if x.nunique() > 1 and sample[column].nunique() > 1 else float("nan")
        row["spearman_change_r"] = x.rank().corr(y.rank()) if x.nunique() > 1 and y.nunique() > 1 else float("nan")
        x_within = x - sample.groupby("ticker")["inventory_gap_pp"].transform("mean")
        y_within = y - sample.groupby("ticker")["outcome_change_pp"].transform("mean")
        row["company_demeaned_r"] = x_within.corr(y_within) if x_within.nunique() > 1 and y_within.nunique() > 1 else float("nan")
        sensitivity_rows.append(row)
sensitivity = pd.DataFrame(sensitivity_rows)
trim_limits = pd.DataFrame(trim_rows)
display(sensitivity.round(3))
display(trim_limits.round(3))


## Cell 8: Save the Exploration Outputs and Research Boundaries

本格保存新的描述性分析表，不覆盖 Module 9 的财务金额、配对表或人工决策文件。
sample_coverage 用于报告样本部分；pooled_summary 与 subgroup_summary 用于描述性结果；subgroup_differences 展示行业内方向。
company_comparisons 保留每家公司及小样本不通过标记，company_balanced_summary 是合格公司等权概要。
sensitivity 和 trim_limits 用于记录所有预定敏感性结果及阈值，避免只展示支持假设的版本。
CSV 保留完整计算精度，前面 .round(3) 只改变显示；后续直接从 CSV 读取原精度数值。
metadata 记录来源模块、权重、分组门槛、敏感性规则及暂定研究边界。

两张图片已在 Cell 4、Cell 5 保存；运行完后保存 Notebook。
Report 现在可以开始整理三段：样本覆盖、库存信号与结果的描述性关系、关系在公司/子行业/样本规则下是否稳定。
先准确陈述数据支持什么，再讨论可能机制；毛利和现金流不必得出相同结论。
下一阶段要核对关键公司个案，并进一步区分基期影响与下一年度表现，再决定是否建立简单回归。


In [ ]:
outputs = {
    "expanded_sample_coverage.csv": sample_coverage,
    "expanded_inventory_group_summary.csv": pooled_summary,
    "expanded_subgroup_group_summary.csv": subgroup_summary,
    "expanded_subgroup_mean_differences.csv": subgroup_differences,
    "expanded_company_group_comparisons.csv": company_comparisons,
    "expanded_company_balanced_summary.csv": company_balanced_summary,
    "expanded_correlation_sensitivity.csv": sensitivity,
    "expanded_inventory_trim_limits.csv": trim_limits,
}
for filename, table in outputs.items():
    table.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "module9_metadata": module9_metadata,
    "sample_status": "provisional", "common_outcome_pairs": len(common_keys),
    "inventory_group_rule": "Gap > 0 versus Gap <= 0; zero belongs to the nonpositive group",
    "within_company_min_pairs_per_group": MIN_GROUP_PAIRS,
    "weighting": "Pooled results weight annual pairs equally; balanced results weight eligible companies equally",
    "sensitivity_rules": ["All pairs", "Unchanged period lengths", "Common outcome pairs", "Central 98% of inventory gaps"],
    "trim_rule": "Outcome-specific 1st-99th inventory-gap percentiles, inclusive; original data retained",
    "correlation_design": "Descriptive Pearson, rank correlation and company-demeaned Pearson; no year adjustment",
    "unit": "Changes in percentage points; ratio levels in percent; rates and correlations in decimals",
    "p_values_computed": False, "causal_identification": False, "prediction_validated": False,
    "accounting_scope_verified": False, "point_in_time_verified": False,
}
(PROCESSED_DIR / "expanded_module10_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Module 10 tables saved in:", PROCESSED_DIR)
print("Module 10 figures saved in:", FIGURE_DIR)
